[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/probability/p01_house_edge.ipynb)

# How casinos and insurers always win

A roulette wheel lets you win almost half the time, yet the casino never loses. Insurers pay out every day, yet most
years they come out ahead. Here you can run the wheel yourself: one player, a thousand players, the race between luck
and the house edge, an insurer with as many customers as you like, and the storm that breaks the whole idea.

**How to use this notebook.** The grey boxes are cells. Click the first one, then press **Shift + Enter** (or the
▶ Run button) to run it. Run each cell in turn, from the top to the bottom. When a slider appears, drag it and watch.
Change any number you like and run the cell again. Nothing can break: if something looks odd, run everything again
from the top. It works in your browser (no install) and in Google Colab.

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look, and a few helpers. Run this cell first.
import math
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, SelectionSlider, ToggleButton

SLATE, SLATE2 = "#1f2b28", "#2a3a36"        # the slate ground
CHALK, DIM = "#ece6d6", "#93a097"           # chalk, and chalk smudge
CORAL = "#e3896b"                           # the one key thing in each figure
TEAL, SAGE, LILAC = "#6fb6bb", "#a0c27c", "#c9a0d3"

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#3b4c47", "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "figure.dpi": 100,
})

P_RED = 18 / 37                             # a European wheel: 18 red pockets out of 37
rng = np.random.default_rng(2026)

def spins(players, n):
    """Each player's money after every spin, betting $1 on red each time (rows = players)."""
    steps = np.where(rng.random((players, n)) < P_RED, 1, -1)
    return np.concatenate([np.zeros((players, 1), int), np.cumsum(steps, axis=1)], axis=1)

print("Ready. Run the next cell.")

## 1 · The house edge, worked out

A dollar on red wins $1 in 18 pockets out of 37 and loses $1 in the other 19. The average result of one play, the
**expected value**, is 18/37 × (+1) + 19/37 × (−1).

In [ ]:
ev = P_RED * 1 + (1 - P_RED) * -1
print(f"expected value of $1 on red: {ev:+.4f} dollars  =  {ev * 100:+.2f} cents a spin")
print(f"an American wheel (38 pockets, two green): {(18 / 38 - 20 / 38) * 100:+.2f} cents a spin")

## 2 · One player, a hundred spins

Run this cell a few times. Each run is a new evening at the table. How often does the player end up ahead?

In [ ]:
path = spins(1, 100)[0]
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.plot(path, color=CORAL, lw=2)
ax.axhline(0, color=DIM, lw=1)
ax.set_xlabel("spins"); ax.set_ylabel("dollars")
ax.set_title(f"after 100 spins: {path[-1]:+d} dollars")
plt.show()

## 3 · A crowd of players

Drag the slider. Each coral line is one player; the green line is the average, sliding down by 1/37 of a dollar a
spin. Watch the fan open while the average holds steady.

In [ ]:
def crowd(players=100):
    n, block = 2000, 20                       # blocks of 20 spins: wins in a block ~ Binomial(20, 18/37), exactly
    steps = 2 * rng.binomial(block, P_RED, (players, n // block)) - block
    m = np.concatenate([np.zeros((players, 1), int), np.cumsum(steps, axis=1)], axis=1)
    x = np.arange(0, n + 1, block)
    fig, ax = plt.subplots(figsize=(9, 4))
    for row in m[:300]:
        ax.plot(x, row, color=CORAL, lw=0.6, alpha=0.25)
    ax.plot(x, m.mean(axis=0), color=SAGE, lw=2.5, label="the players' average")
    ax.plot(x, -x / 37, color=CHALK, lw=1, ls="--", label="the edge: −n/37")
    ax.axhline(0, color=DIM, lw=1)
    ax.set_xlabel("spins"); ax.set_ylabel("dollars"); ax.legend(loc="lower left")
    ahead = (m[:, -1] > 0).mean() * 100
    ax.set_title(f"{players:,} players · {ahead:.1f}% ahead after {n:,} spins")
    plt.show()

interact(crowd, players=SelectionSlider(options=[1, 3, 10, 30, 100, 300, 1000, 3000, 10000], value=1, description="players"));

## 4 · The race: luck against the edge

After n spins, the house's pull is n/37 dollars, and a typical swing of luck is about √n dollars. The pull grows
like n; luck only like √n. Where do they cross?

In [ ]:
n = np.arange(0, 3001)
pull = n / 37
luck = np.sqrt(n) * math.sqrt(1 - 1 / 37 ** 2)
cross = int(round((37 * math.sqrt(1 - 1 / 37 ** 2)) ** 2))
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(n, luck, color=CORAL, lw=2.5, label="luck: a typical swing")
ax.plot(n, pull, color=SAGE, lw=2.5, label="the pull: n / 37")
ax.axvline(cross, color=CHALK, lw=1, ls="--")
ax.set_xlabel("spins"); ax.set_ylabel("dollars"); ax.legend()
ax.set_title(f"they cross at {cross:,} spins  (37 × 37 = {37 * 37:,})")
plt.show()

## 5 · Does luck even out?

Suppose black has just come up 26 times in a row. Keep spinning fairly. Red never "pays it back": the streak is just
drowned by everything after it. Pick how many spins to add.

In [ ]:
def drown(more=1000):
    after = (rng.random(more) >= 1 - 18 / 37)          # True = black (18 pockets in 37)
    blacks = 26 + np.cumsum(after)
    total = 26 + np.arange(1, more + 1)
    fig, ax = plt.subplots(figsize=(9, 3.6))
    ax.plot(total, blacks / total, color=CHALK, lw=2)
    ax.axhline(18 / 37, color=SAGE, lw=1.5, label="fair share 48.65%")
    ax.set_xscale("log"); ax.set_xlabel("spins since the run began"); ax.set_ylabel("share of black")
    ax.legend(); ax.set_title(f"after {total[-1]:,} spins: {blacks[-1] / total[-1] * 100:.2f}% black")
    plt.show()

interact(drown, more=IntSlider(value=1000, min=10, max=200_000, step=10, description="more spins"));

## 6 · An insurer

Made-up but realistic numbers: one phone in ten needs a $200 repair each year, so the expected claim is $20; the
insurer charges $25. Drag the customers slider and see how often a year loses money. Then switch on **the storm**:
every phone breaks together, or none do.

In [ ]:
def insurer(customers=10, storm=False):
    years = 2000
    if storm:
        broken = np.where(rng.random(years) < 0.1, customers, 0)
    else:
        broken = rng.binomial(customers, 0.1, years)
    result = 25 * customers - 200 * broken
    fig, ax = plt.subplots(figsize=(9, 3.6))
    ax.hist(result, bins=60, color=LILAC)
    ax.axvline(0, color=CORAL, lw=2)
    ax.set_xlabel("the insurer's result for a year (dollars)"); ax.set_ylabel("years")
    ax.set_title(f"{customers:,} customers{' · STORM' if storm else ''}: a losing year {(result < 0).mean() * 100:.1f}% of the time")
    plt.show()

interact(insurer, customers=SelectionSlider(options=[10, 100, 1000, 10000, 100000], value=10, description="customers"),
         storm=ToggleButton(value=False, description="the storm"));

## 7 · The challenge from the video

An American wheel has 38 pockets, two of them green. After about how many spins does its pull catch up with a
typical swing of luck? Think first, then press the button.

In [ ]:
def challenge(show_answer=False):
    if not show_answer:
        print("An American wheel: after how many spins does n/19 catch up with √n?   (press the button)"); return
    print("A dollar on red loses 2/38 = 1/19 on average, so the pull is n/19.")
    print(f"n/19 = √n  →  n = 19² = {19 ** 2}  (exactly {round((19 * math.sqrt(1 - 1 / 19 ** 2)) ** 2)} with the swing's"
          " small correction)")

interact(challenge, show_answer=ToggleButton(value=False, description="show the answer"));

## Sources
Pascal's letter to Fermat, 29 July 1654; Huygens, *De ratiociniis in ludo aleae* (1657); Jacob Bernoulli, *Ars
Conjectandi* (1713); Halley, *Phil. Trans.* 17 (1693) 596–610. The phone insurer is an illustration. Insurance
industry figures in the video are estimates (AM Best; S&P Global Market Intelligence). All retrieved 2026-11-15.
Every number above is computed in the cells. Nothing here is betting advice.